In [1]:
# 本次agent基于react模式，即reasoning和action

In [2]:
# 获得环境变量，创建客户端
import os
import re

from openai import OpenAI, api_key

In [3]:
client = OpenAI(
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/"
)

In [4]:
chat_completion = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[{"role": "user", "content": "Hello World"}]
)

In [5]:
chat_completion.choices[0].message.content

'Hello! 👋 How can I help you today?'

In [6]:
class Agent:
    # 允许用户传入数据
    def __init__(self, system=""):
        self.system = system
        # 记录react循环当中的所有消息
        self.messages = []
        if self.system :
             self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        # 接收传入的消息
        self.messages.append({"role": "user", "content": message})
        result = self.execute()

        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model="deepseek-v4-flash",
            temperature=0,
            messages=self.messages
        )
        # 返回模型输出的回复
        return completion.choices[0].message.content

In [7]:
# 创建ReAct智能体
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [8]:
# 创建所需的工具
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier":
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")


known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [9]:
abot = Agent(prompt) # prompt指示的是system_message

In [10]:
result = abot("How much does a toy poodle weigh?")
print(result)

Thought: I should look up the average weight of a Toy Poodle using the dog weight tool.
Action: average_dog_weight: Toy Poodle
PAUSE


In [11]:
result = average_dog_weight("Toy Poodle")

In [12]:
result

'a toy poodles average weight is 7 lbs'

In [13]:
next_prompt = "Observation: {}".format(result)

In [14]:
next_prompt

'Observation: a toy poodles average weight is 7 lbs'

In [15]:
abot(next_prompt)

'Answer: A Toy Poodle weighs about 7 lbs on average.'

In [16]:
abot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user', 'content': 'How much does a 

In [17]:
abot = Agent(prompt) # 传入prompt,使用不同的示例

In [18]:
question = """I have 2 dogs, a border collie and a scottish terrier. What is their combined weight"""

abot(question)

'Thought: I need the average weight of each breed, then add them together.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [19]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))

In [20]:
print(next_prompt)

Observation: a Border Collies average weight is 37 lbs


In [21]:
abot(next_prompt)

"Thought: Now I need the Scottish Terrier's average weight.\nAction: average_dog_weight: Scottish Terrier\nPAUSE"

In [22]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))

In [23]:
abot(next_prompt)

'Thought: I now have both weights, so I should add them together.\nAction: calculate: 37 + 20\nPAUSE'

In [24]:
next_prompt = "Observation: {}".format(eval("37 + 20"))

In [25]:
print(next_prompt)

Observation: 57


In [26]:
abot(next_prompt)

'Answer: Their combined weight is 57 lbs.'

In [27]:
abot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user',
  'content': 'I have 2 dogs,

In [28]:
# 创建正则表达式
import re

action_re  = re.compile('^Action: (\w+): (.*)$')

<>:4: SyntaxWarning: invalid escape sequence '\w'
<>:4: SyntaxWarning: invalid escape sequence '\w'
C:\Users\hilary\AppData\Local\Temp\ipykernel_15252\1078002538.py:4: SyntaxWarning: invalid escape sequence '\w'
  action_re  = re.compile('^Action: (\w+): (.*)$')


In [29]:
def query(question, max_turns=5):
    # 定义一个计数器
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]

        if actions:
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation =known_actions[action](action_input)
        else:
            return

In [30]:
query(question)

Thought: I need the average weights of both breeds, then add them together.
Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
Thought: I need the average weights of both breeds. I’ll start by looking up the Border Collie’s average weight.
Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
A Border Collie averages about **35–45 lbs**, and a Scottish Terrier averages about **18–22 lbs**.

So their combined weight is roughly **55–65 lbs**, with about **60 lbs** as a typical estimate.
